# Highland Square Conjoint — Notebook 3: Interactive Market-Share Simulator

## What this notebook does

Builds an interactive simulator that takes the estimated coefficients from Notebook 2 and lets you:
1. **Configure Highland Square** (the subject) and competing properties feature-by-feature
2. **See market share** for each persona AND population-weighted overall
3. **View persona descriptions** to understand what each segment values
4. **Calculate WTP** for any feature upgrade in dollars-per-month terms
5. **Test repositioning scenarios** — e.g., "if I add a rooftop and bump rent $200, do I gain or lose share?"

## How to use this for IC underwriting

**Repositioning case:** Build a current-state Highland Square plus the top 2-3 competing comps. Note the baseline shares. Then upgrade Highland Square (premium finishes, security tier, rooftop, etc.) and bump rent. The simulator tells you whether the rent bump is supportable — if share holds or grows, the WTP supports the rent. If share collapses, you're overshooting.

**Capture-rate underwriting:** Population-weighted share × total qualified renter pool (from ACS / AxleData for the trade area) = projected lease-up velocity. The persona weights matter — adjust them to match the actual demographic mix of your trade area.

**Rent positioning:** Hold all features constant and sweep price levels ($1,650 / $1,950 / $2,250 / $2,550). Plot share vs. price. The slope is the elasticity of demand at your asset's positioning.

## 0. Setup

In [1]:
!pip install ipywidgets --quiet

In [2]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Markdown

warnings.filterwarnings('ignore')
DATA_DIR = Path('data')
print("Setup complete.")

Setup complete.


## 1. Load Estimated Coefficients from Notebook 2

In [3]:
# Load ATTRIBUTES dict
with open(DATA_DIR / 'attributes.json') as f:
    ATTRIBUTES = json.load(f)

# Load per-persona coefficients
with open(DATA_DIR / 'persona_coefs.json') as f:
    persona_coefs_raw = json.load(f)
persona_coefs = {p: pd.DataFrame(rows) for p, rows in persona_coefs_raw.items()}

# Load pooled coefficients
pooled_coefs = pd.read_csv(DATA_DIR / 'pooled_coefs.csv')

attr_names = list(ATTRIBUTES.keys())
CONTINUOUS_ATTRS = [a for a in attr_names if any(v is not None for v in ATTRIBUTES[a].values())]
CATEGORICAL_ATTRS = [a for a in attr_names if a not in CONTINUOUS_ATTRS]

print(f"Loaded {len(persona_coefs)} persona models: {list(persona_coefs.keys())}")
print(f"Attributes: {len(attr_names)} ({len(CATEGORICAL_ATTRS)} categorical + {len(CONTINUOUS_ATTRS)} continuous)")

Loaded 4 persona models: ['emory_grad', 'vahi_professional', 'empty_nester', 'skeptical_renter_control']
Attributes: 13 (11 categorical + 2 continuous)


## 2. Persona Descriptions

In [4]:
PERSONA_DESCRIPTIONS = {
    "emory_grad": {
        "name": "Maya (Emory PhD Candidate)",
        "age": 28,
        "income": "$52,000/year (stipend + RA funding)",
        "savings": "$4,200 liquid",
        "debt": "$580/mo federal student loans",
        "work_destination": "Emory main campus",
        "lease_horizon": "18+ months (through dissertation)",
        "segment": "Budget-constrained academic professional",
        "narrative": "Maya is a third-year PhD candidate in epidemiology. Her monthly take-home is roughly $3,200, and she's loan-burdened. She's price-sensitive and values stability through her dissertation timeline. She doesn't have wealth buffers — she lives close to her budget every month.",
    },
    "young_professional": {
        "name": "David (Healthcare Consultant)",
        "age": 34,
        "income": "$135K base + $20K bonus = $155K",
        "savings": "$42K HYSA + $185K 401(k)",
        "debt": "$14K residual student loans",
        "work_destination": "Midtown (W. Peachtree), 3 days/week",
        "lease_horizon": "2-3 years",
        "segment": "Mid-career hybrid professional",
        "narrative": "David is a single, well-compensated consultant new to Atlanta. He's not budget-constrained but values getting his money's worth. Hybrid work means apartment quality matters — he's home 2 days a week. Likely to upgrade if the value proposition is clear.",
    },
    "vahi_professional": {
        "name": "David (Healthcare Consultant)",
        "age": 34,
        "income": "$135K base + $20K bonus = $155K",
        "savings": "$42K HYSA + $185K 401(k)",
        "debt": "$14K residual student loans",
        "work_destination": "Midtown (W. Peachtree), 3 days/week",
        "lease_horizon": "2-3 years",
        "segment": "Mid-career hybrid professional",
        "narrative": "David is a single, well-compensated consultant new to Atlanta. He's not budget-constrained but values getting his money's worth. Hybrid work means apartment quality matters — he's home 2 days a week. Likely to upgrade if the value proposition is clear.",
    },
    "empty_nester": {
        "name": "Patricia (Recent Retiree)",
        "age": 58,
        "income": "$180K household (Tom still works as CPA)",
        "savings": "~$1.4M retirement + $815K home-sale proceeds",
        "debt": "None",
        "work_destination": "Sandy Springs (Tom's office)",
        "lease_horizon": "2-4 years (then condo TBD)",
        "segment": "Wealthy empty-nester downsizer",
        "narrative": "Patricia and Tom sold their suburban house and are renting for the first time in 30 years. They have substantial home-sale proceeds and retirement assets. Price is not a major constraint — they value quality, security, and the right neighborhood. Tom still commutes, so commute time matters for him.",
    },
    "skeptical_renter_control": {
        "name": "Alex (Software Engineer)",
        "age": 31,
        "income": "$115K",
        "savings": "$35K HYSA + $95K 401(k)",
        "debt": "None",
        "work_destination": "Unspecified",
        "lease_horizon": "1-2 years",
        "segment": "Analytical comparison shopper",
        "narrative": "Alex is an experienced renter in Atlanta — has lived in Midtown, West Midtown, and Decatur. Reads reviews carefully and would walk away from a bad deal. Serves as a control persona to anchor the analytical end of the renter spectrum.",
    },
}

# Show summary table
rows = []
for k in persona_coefs.keys():
    if k in PERSONA_DESCRIPTIONS:
        d = PERSONA_DESCRIPTIONS[k]
        rows.append({'key': k, 'Name': d['name'], 'Income': d['income'],
                     'Savings': d['savings'], 'Segment': d['segment']})
personas_summary = pd.DataFrame(rows)
personas_summary

,key,Name,Income,Savings,Segment
0,emory_grad,Maya (Emory PhD Candidate),"$52,000/year (stipend + RA funding)","$4,200 liquid",Budget-constrained academic professional
1,vahi_professional,David (Healthcare Consultant),$135K base + $20K bonus = $155K,$42K HYSA + $185K 401(k),Mid-career hybrid professional
2,empty_nester,Patricia (Recent Retiree),$180K household (Tom still works as CPA),~$1.4M retirement + $815K home-sale proceeds,Wealthy empty-nester downsizer
3,skeptical_renter_control,Alex (Software Engineer),$115K,$35K HYSA + $95K 401(k),Analytical comparison shopper


## 3. Utility & Market-Share Functions

In [5]:
def property_utility(profile, coefs):
    """Compute deterministic utility of a profile under a given coef vector.
    
    profile: dict with all 13 attributes set to a valid level label
    coefs: DataFrame with 'feature' and 'coef' columns (from persona_coefs or pooled)
    """
    util = 0.0
    # Continuous attrs
    util += coefs.loc[coefs['feature'] == 'Size_num', 'coef'].values[0] * ATTRIBUTES['Size'][profile['Size']]
    util += coefs.loc[coefs['feature'] == 'Price_num', 'coef'].values[0] * ATTRIBUTES['Price'][profile['Price']]
    # Categorical dummies (baseline = first level, contributes 0)
    for attr in attr_names:
        if attr in ('Size', 'Price'):
            continue
        level = profile[attr]
        baseline = list(ATTRIBUTES[attr].keys())[0]
        if level == baseline:
            continue
        feat = f"{attr}__{level}"
        match = coefs.loc[coefs['feature'] == feat, 'coef']
        if len(match) > 0:
            util += match.values[0]
    return util

def market_share(competing_profiles, coefs):
    """Softmax over competing properties to compute conditional choice probabilities."""
    utils = np.array([property_utility(p, coefs) for p in competing_profiles])
    exp_u = np.exp(utils - utils.max())
    shares = exp_u / exp_u.sum()
    return shares

def wtp_for_change(attribute, from_level, to_level, coefs):
    """Compute WTP ($/mo) for upgrading attribute from `from_level` to `to_level`.
    
    For continuous attrs (Size), pass numeric values.
    For categorical, pass level labels.
    """
    bp = coefs.loc[coefs['feature'] == 'Price_num', 'coef'].values[0]
    if bp >= 0:
        return np.nan
    if attribute == 'Size':
        bs = coefs.loc[coefs['feature'] == 'Size_num', 'coef'].values[0]
        return -bs * (to_level - from_level) / bp
    
    # Categorical: difference in dummies
    baseline = list(ATTRIBUTES[attribute].keys())[0]
    def coef_for_level(lvl):
        if lvl == baseline:
            return 0.0
        m = coefs.loc[coefs['feature'] == f"{attribute}__{lvl}", 'coef']
        return m.values[0] if len(m) > 0 else 0.0
    delta_util = coef_for_level(to_level) - coef_for_level(from_level)
    return -delta_util / bp

print("Functions defined.")

Functions defined.


## 4. Default Property Profiles

Two starter profiles: Highland Square (current state) and Modera Morningside (the key competing comp). You'll edit these via widgets below.

In [6]:
HIGHLAND_SQUARE_CURRENT = {
    'Name': 'Highland Square (current)',
    'Size': '1,000 SF (large 1BR / compact 2BR)',
    'Price': '$1,950/mo',
    'MoveInSpecial': '1 month free (12-mo lease)',
    'Location': 'North Druid Hills / Briarcliff',
    'CommuteToWork': 'Average (15-30 min by car)',
    'Walkability': 'Walkable Errands (groceries & a few restaurants within a 10-min walk of this building)',
    'Finishes': 'Mid-tier (granite/quartz counters, stainless appliances, in-unit washer/dryer)',
    'Parking': 'Gated surface lot + reserved space option',
    'Security': 'Tier 2: Perimeter gate + controlled-access lobby + camera coverage',
    'Rooftop': 'No rooftop space',
    'Coworking': 'No dedicated coworking space',
    'PetAmenities': 'Standard dog park only',
    'PackageHandling': 'Standard mailroom (sign for packages during office hours)',
}

MODERA_COMP = {
    'Name': 'Modera Morningside (key comp)',
    'Size': '1,000 SF (large 1BR / compact 2BR)',
    'Price': '$2,250/mo',
    'MoveInSpecial': 'None',
    'Location': 'Virginia-Highland / Morningside',
    'CommuteToWork': 'Average (15-30 min by car)',
    'Walkability': 'Walk Everywhere (daily errands, dining, transit within a 10-min walk of this building)',
    'Finishes': 'Premium (quartz waterfall island, smart thermostat, keyless entry, video doorbell)',
    'Parking': 'Dedicated garage with assigned space + EV charging',
    'Security': 'Tier 3: Tier 2 + 24/7 staff or virtual concierge + smart locks throughout',
    'Rooftop': 'Rooftop lounge with skyline views & outdoor seating',
    'Coworking': 'Resident co-working lounge with private call rooms & wifi',
    'PetAmenities': 'Dog park + pet spa with grooming station',
    'PackageHandling': '24/7 Amazon Hub lockers + refrigerated grocery locker',
}

print("Default profiles defined.")

Default profiles defined.


## 5. Interactive Property Configurator

Configure Highland Square (Property 1) and a competitor (Property 2). Each attribute has a dropdown. Run the next cell to see updated market share.

In [7]:
def make_property_widgets(profile, name='Property'):
    """Build a dropdown widget for every attribute of one property."""
    wdict = {}
    name_w = widgets.Text(value=profile.get('Name', name), description='Name:', layout=widgets.Layout(width='400px'))
    wdict['Name'] = name_w
    for attr in attr_names:
        levels = list(ATTRIBUTES[attr].keys())
        wdict[attr] = widgets.Dropdown(
            options=levels,
            value=profile[attr],
            description=attr + ':',
            layout=widgets.Layout(width='700px'),
            style={'description_width': '120px'},
        )
    return wdict

def widgets_to_profile(wdict):
    return {k: w.value for k, w in wdict.items()}

# Build widget sets for both properties
prop1_widgets = make_property_widgets(HIGHLAND_SQUARE_CURRENT, 'Highland Square')
prop2_widgets = make_property_widgets(MODERA_COMP, 'Competitor')

prop1_box = widgets.VBox(list(prop1_widgets.values()), layout=widgets.Layout(border='2px solid #1d4ed8', padding='10px'))
prop2_box = widgets.VBox(list(prop2_widgets.values()), layout=widgets.Layout(border='2px solid #dc2626', padding='10px'))

props_box = widgets.HBox([
    widgets.VBox([widgets.HTML("<h3 style='color:#1d4ed8'>Property 1 (Subject)</h3>"), prop1_box]),
    widgets.VBox([widgets.HTML("<h3 style='color:#dc2626'>Property 2 (Competitor)</h3>"), prop2_box]),
])
display(props_box)

## 6. Population Weights (adjust for your trade area)

In [8]:
# Default weights — replace with ACS / AxleData demographics for the trade area
DEFAULT_WEIGHTS = {
    "emory_grad": 0.20,
    "young_professional": 0.40,
    "vahi_professional": 0.40,  # accommodate both naming conventions
    "empty_nester": 0.20,
    "skeptical_renter_control": 0.20,
}

# Build weight sliders for ONLY the personas that exist in persona_coefs
available_personas = list(persona_coefs.keys())
weight_widgets = {}
for p in available_personas:
    label = PERSONA_DESCRIPTIONS.get(p, {}).get('name', p)
    weight_widgets[p] = widgets.FloatSlider(
        value=DEFAULT_WEIGHTS.get(p, 1.0/len(available_personas)),
        min=0.0, max=1.0, step=0.05,
        description=f"{label}:",
        style={'description_width': '320px'},
        layout=widgets.Layout(width='700px'),
    )

weights_box = widgets.VBox([
    widgets.HTML("<h3>Population Weights (will be normalized to sum to 1.0)</h3>"),
    widgets.HTML("<p style='color:#555;font-size:13px'>Adjust to match the demographic mix in your trade area. Pull from ACS B25010, AxleData, or local broker data.</p>"),
    *weight_widgets.values()
])
display(weights_box)

## 7. Run Simulation — Hit "Simulate" After Adjusting Widgets Above

In [9]:
simulate_btn = widgets.Button(description='Run Simulation', button_style='primary',
                              layout=widgets.Layout(width='200px', height='40px'))
results_output = widgets.Output()

def run_sim(b=None):
    with results_output:
        clear_output()
        prop1 = widgets_to_profile(prop1_widgets)
        prop2 = widgets_to_profile(prop2_widgets)
        competing = [prop1, prop2]
        names = [prop1['Name'], prop2['Name']]
        
        # Per-persona shares
        persona_shares = {}
        for persona, pdf in persona_coefs.items():
            bp = pdf.loc[pdf['feature'] == 'Price_num', 'coef'].values[0]
            if bp >= 0:
                continue  # skip personas with non-negative price coef
            shares = market_share(competing, pdf)
            persona_shares[persona] = shares
        
        # Population-weighted share
        raw_weights = {p: weight_widgets[p].value for p in persona_shares.keys()}
        wsum = sum(raw_weights.values())
        if wsum <= 0:
            print("⚠ All weights are 0. Set at least one weight > 0.")
            return
        norm_weights = {p: w/wsum for p, w in raw_weights.items()}
        
        weighted = np.zeros(len(competing))
        for persona, shares in persona_shares.items():
            weighted += norm_weights[persona] * shares
        
        # Display: per-persona table
        print("=" * 78)
        print("MARKET SHARE BY PERSONA (conditional on choosing to lease)")
        print("=" * 78)
        rows = []
        for persona, shares in persona_shares.items():
            display_name = PERSONA_DESCRIPTIONS.get(persona, {}).get('name', persona)
            rows.append({
                'Persona': display_name,
                'Weight': f"{norm_weights[persona]*100:.0f}%",
                names[0]: f"{shares[0]*100:.1f}%",
                names[1]: f"{shares[1]*100:.1f}%",
            })
        share_df = pd.DataFrame(rows)
        print(share_df.to_string(index=False))
        
        print("\n" + "=" * 78)
        print("POPULATION-WEIGHTED OVERALL SHARE")
        print("=" * 78)
        for i, name in enumerate(names):
            print(f"  {name:50s}  {weighted[i]*100:5.1f}%")
        
        # Bar chart
        fig, ax = plt.subplots(figsize=(11, 5))
        n_personas = len(persona_shares)
        x = np.arange(2)  # 2 properties
        width = 0.8 / (n_personas + 1)
        for i, (persona, shares) in enumerate(persona_shares.items()):
            label = PERSONA_DESCRIPTIONS.get(persona, {}).get('name', persona)
            ax.bar(x + i*width, shares*100, width, label=label, alpha=0.75)
        ax.bar(x + n_personas*width, weighted*100, width, label='Population-weighted',
               color='black', alpha=0.85)
        ax.set_xticks(x + (n_personas)*width/2)
        ax.set_xticklabels([n[:35] for n in names], rotation=0, fontsize=10)
        ax.set_ylabel('Market Share (%)', fontsize=11)
        ax.set_title('Market Share by Persona + Population-Weighted', fontsize=12, fontweight='bold')
        ax.legend(loc='upper right', fontsize=9)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.show()
        
        print("\n💡 IC Memo interpretation:")
        share_subject = weighted[0]*100
        share_comp = weighted[1]*100
        if share_subject > share_comp:
            print(f"  Subject captures {share_subject:.0f}% vs competitor's {share_comp:.0f}% under current weights.")
            print(f"  At ~30% capture of a ~3,000-unit trade-area renter pool, expected captured renter count = {0.3 * 3000:.0f}.")
        else:
            print(f"  Subject captures only {share_subject:.0f}% vs competitor's {share_comp:.0f}%.")
            print(f"  Consider repositioning: see the WTP table below for which upgrades carry the highest WTP per dollar.")

simulate_btn.on_click(run_sim)
display(simulate_btn, results_output)
run_sim()  # initial run

Button(button_style='primary', description='Run Simulation', layout=Layout(height='40px', width='200px'), styl…

Output()

## 8. Persona Descriptions Viewer

In [10]:
persona_selector = widgets.Dropdown(
    options=list(persona_coefs.keys()),
    description='Persona:',
    layout=widgets.Layout(width='400px'),
)
persona_output = widgets.Output()

def show_persona(change=None):
    with persona_output:
        clear_output()
        k = persona_selector.value
        if k not in PERSONA_DESCRIPTIONS:
            print(f"No description available for {k}")
            return
        d = PERSONA_DESCRIPTIONS[k]
        # Get coefficients
        pdf = persona_coefs[k]
        bp = pdf.loc[pdf['feature'] == 'Price_num', 'coef'].values[0]
        
        html = f"""
        <div style='font-family:Arial;padding:14px;border:2px solid #1d4ed8;border-radius:8px;background:#f0f9ff'>
        <h3 style='color:#1d4ed8;margin-top:0'>{d['name']}</h3>
        <table style='font-size:13px'>
        <tr><td><b>Age:</b></td><td>{d['age']}</td></tr>
        <tr><td><b>Income:</b></td><td>{d['income']}</td></tr>
        <tr><td><b>Savings:</b></td><td>{d['savings']}</td></tr>
        <tr><td><b>Debt:</b></td><td>{d['debt']}</td></tr>
        <tr><td><b>Daily destination:</b></td><td>{d['work_destination']}</td></tr>
        <tr><td><b>Lease horizon:</b></td><td>{d['lease_horizon']}</td></tr>
        <tr><td><b>Segment:</b></td><td>{d['segment']}</td></tr>
        <tr><td><b>Price sensitivity (β):</b></td><td>{bp:.5f} ({'high' if abs(bp) > 0.01 else 'moderate' if abs(bp) > 0.005 else 'low'})</td></tr>
        </table>
        <p style='margin-top:10px'><b>Narrative:</b> {d['narrative']}</p>
        </div>
        """
        display(HTML(html))
        
        # Top 5 things this persona values most (positive WTP)
        wtp_for_persona = []
        for _, row in pdf.iterrows():
            feat = row['feature']
            if feat in ('Size_num', 'Price_num', 'ASC_B', 'ASC_C'):
                continue
            if '__' not in feat:
                continue
            wtp = -row['coef'] / bp
            wtp_for_persona.append({'feature': feat, 'WTP_$/mo': wtp})
        wp = pd.DataFrame(wtp_for_persona).sort_values('WTP_$/mo', ascending=False)
        
        print(f"\nTop 5 highest WTP for {d['name']}:")
        for _, r in wp.head(5).iterrows():
            print(f"  ${r['WTP_$/mo']:7.0f}/mo  {r['feature'][:80]}")
        print(f"\nTop 5 lowest WTP (most disliked):")
        for _, r in wp.tail(5).iterrows():
            print(f"  ${r['WTP_$/mo']:7.0f}/mo  {r['feature'][:80]}")

persona_selector.observe(show_persona, names='value')
display(persona_selector, persona_output)
show_persona()  # initial display

Dropdown(description='Persona:', layout=Layout(width='400px'), options=('emory_grad', 'vahi_professional', 'em…

Output()

## 9. WTP Calculator: Pick Any Two Feature Levels to Compare

In [11]:
wtp_attr = widgets.Dropdown(options=[a for a in attr_names if a != 'Price'], description='Attribute:',
                            layout=widgets.Layout(width='400px'))
wtp_from = widgets.Dropdown(options=list(ATTRIBUTES[wtp_attr.value].keys()), description='From level:',
                            layout=widgets.Layout(width='700px'))
wtp_to = widgets.Dropdown(options=list(ATTRIBUTES[wtp_attr.value].keys()), description='To level:',
                          layout=widgets.Layout(width='700px'))
wtp_output = widgets.Output()

def update_levels(change=None):
    wtp_from.options = list(ATTRIBUTES[wtp_attr.value].keys())
    wtp_to.options = list(ATTRIBUTES[wtp_attr.value].keys())
    show_wtp()

def show_wtp(change=None):
    with wtp_output:
        clear_output()
        attr = wtp_attr.value
        from_lvl = wtp_from.value
        to_lvl = wtp_to.value
        if from_lvl == to_lvl:
            print("Pick different from/to levels.")
            return
        print(f"WTP for changing {attr} from")
        print(f"  '{from_lvl}'")
        print(f"to")
        print(f"  '{to_lvl}'")
        print()
        
        if attr == 'Size':
            sf_from = ATTRIBUTES['Size'][from_lvl]
            sf_to = ATTRIBUTES['Size'][to_lvl]
            print(f"  (Size change: {sf_from} → {sf_to} SF, Δ = {sf_to - sf_from} SF)\n")
        
        # Pooled WTP
        if attr == 'Size':
            wtp_p = wtp_for_change(attr, sf_from, sf_to, pooled_coefs)
        else:
            wtp_p = wtp_for_change(attr, from_lvl, to_lvl, pooled_coefs)
        print(f"  Pooled (avg renter):     ${wtp_p:+8.0f}/mo")
        
        # Per-persona WTP
        for persona, pdf in persona_coefs.items():
            bp = pdf.loc[pdf['feature'] == 'Price_num', 'coef'].values[0]
            if bp >= 0:
                continue
            if attr == 'Size':
                w = wtp_for_change(attr, sf_from, sf_to, pdf)
            else:
                w = wtp_for_change(attr, from_lvl, to_lvl, pdf)
            label = PERSONA_DESCRIPTIONS.get(persona, {}).get('name', persona)[:30]
            print(f"  {label:30s}  ${w:+8.0f}/mo")
        
        print("\nInterpretation:")
        if wtp_p > 0:
            print(f"  The avg renter would pay an extra ${wtp_p:.0f}/month for this upgrade.")
            print(f"  If renovating to this level costs $X total, payback period = $X / ${wtp_p*12:.0f}/year.")
        else:
            print(f"  The avg renter actually PREFERS the 'from' level. Moving 'to' = -${-wtp_p:.0f}/mo rent loss.")

wtp_attr.observe(update_levels, names='value')
wtp_from.observe(show_wtp, names='value')
wtp_to.observe(show_wtp, names='value')

display(wtp_attr, wtp_from, wtp_to, wtp_output)
show_wtp()

Dropdown(description='Attribute:', layout=Layout(width='400px'), options=('Size', 'MoveInSpecial', 'Location',…

Dropdown(description='From level:', layout=Layout(width='700px'), options=('750 SF (compact 1BR)', '1,000 SF (…

Dropdown(description='To level:', layout=Layout(width='700px'), options=('750 SF (compact 1BR)', '1,000 SF (la…

Output()

## 10. Repositioning Scenario Tester

Bulk-apply common repositioning bundles and see the impact on Highland Square's share.

In [12]:
REPOSITIONING_BUNDLES = {
    "Light renovation (mid-tier finishes + Tier 2 security)": {
        'Finishes': 'Mid-tier (granite/quartz counters, stainless appliances, in-unit washer/dryer)',
        'Security': 'Tier 2: Perimeter gate + controlled-access lobby + camera coverage',
        'price_bump': 100,
    },
    "Premium repositioning (premium finishes + Tier 3 security + rooftop)": {
        'Finishes': 'Premium (quartz waterfall island, smart thermostat, keyless entry, video doorbell)',
        'Security': 'Tier 3: Tier 2 + 24/7 staff or virtual concierge + smart locks throughout',
        'Rooftop': 'Rooftop lounge with skyline views & outdoor seating',
        'price_bump': 300,
    },
    "Lifestyle pack (coworking + pet spa + smart package)": {
        'Coworking': 'Resident co-working lounge with private call rooms & wifi',
        'PetAmenities': 'Dog park + pet spa with grooming station',
        'PackageHandling': '24/7 Amazon Hub lockers + refrigerated grocery locker',
        'price_bump': 150,
    },
    "Parking upgrade only (gated garage + EV)": {
        'Parking': 'Dedicated garage with assigned space + EV charging',
        'price_bump': 75,
    },
    "Aggressive concession (no other change)": {
        'MoveInSpecial': '2 months free (13-mo lease)',
        'price_bump': 0,
    },
}

scenario_dropdown = widgets.Dropdown(
    options=list(REPOSITIONING_BUNDLES.keys()),
    description='Bundle:',
    layout=widgets.Layout(width='600px'),
    style={'description_width': '80px'},
)
scenario_output = widgets.Output()

def run_scenario(change=None):
    with scenario_output:
        clear_output()
        bundle = REPOSITIONING_BUNDLES[scenario_dropdown.value]
        
        # Build before-repositioning property
        before = widgets_to_profile(prop1_widgets).copy()
        before['Name'] = 'Before repositioning'
        
        # Apply bundle
        after = before.copy()
        after['Name'] = 'After repositioning'
        for k, v in bundle.items():
            if k == 'price_bump':
                continue
            after[k] = v
        # Bump price
        current_price = ATTRIBUTES['Price'][before['Price']]
        target_price = current_price + bundle['price_bump']
        # Find closest available price level
        price_levels = sorted(ATTRIBUTES['Price'].items(), key=lambda x: x[1])
        closest = min(price_levels, key=lambda x: abs(x[1] - target_price))
        after['Price'] = closest[0]
        
        # Competitor unchanged
        comp = widgets_to_profile(prop2_widgets)
        
        # Run market share for before & after
        for label, subject in [('BEFORE', before), ('AFTER', after)]:
            print(f"\n=== {label}: Highland Square = {subject.get('Price')} ===")
            for k, v in bundle.items():
                if k != 'price_bump':
                    print(f"    {k}: {subject[k][:60]}")
            print()
            
            competing = [subject, comp]
            weighted = np.zeros(2)
            raw_w = {p: weight_widgets[p].value for p in persona_coefs.keys()}
            wsum = sum(raw_w.values())
            for persona, pdf in persona_coefs.items():
                bp = pdf.loc[pdf['feature'] == 'Price_num', 'coef'].values[0]
                if bp >= 0:
                    continue
                shares = market_share(competing, pdf)
                weighted += (raw_w[persona] / wsum) * shares
            
            print(f"    Subject share:    {weighted[0]*100:.1f}%")
            print(f"    Competitor share: {weighted[1]*100:.1f}%")
        
        # Delta
        comp_list_before = [before, comp]
        comp_list_after = [after, comp]
        
        before_share = 0
        after_share = 0
        for persona, pdf in persona_coefs.items():
            bp = pdf.loc[pdf['feature'] == 'Price_num', 'coef'].values[0]
            if bp >= 0:
                continue
            before_share += (raw_w[persona]/wsum) * market_share(comp_list_before, pdf)[0]
            after_share += (raw_w[persona]/wsum) * market_share(comp_list_after, pdf)[0]
        
        delta = (after_share - before_share) * 100
        rent_delta = ATTRIBUTES['Price'][after['Price']] - ATTRIBUTES['Price'][before['Price']]
        
        print(f"\n📊 Impact: share Δ = {delta:+.1f}pp, rent Δ = ${rent_delta:+.0f}/mo")
        if delta >= 0 and rent_delta > 0:
            print(f"   ✓ Repositioning supports the rent bump — share holds/grows.")
        elif delta >= 0 and rent_delta == 0:
            print(f"   ✓ Free upside — share grew without rent change.")
        elif delta < 0 and rent_delta > 0:
            print(f"   ✗ Repositioning does NOT support this rent bump — share falls.")
        else:
            print(f"   ⚠ Share fell despite no rent change — check the bundle.")

scenario_dropdown.observe(run_scenario, names='value')
display(scenario_dropdown, scenario_output)
run_scenario()

Dropdown(description='Bundle:', layout=Layout(width='600px'), options=('Light renovation (mid-tier finishes + …

Output()

## 11. How to Use This for IC Underwriting

### Workflow A — Repositioning Decision
1. Set Property 1 to current Highland Square; set Property 2 to your top comp.
2. Note baseline share (Section 7).
3. Open Section 10 "Repositioning Scenario Tester." Try each bundle. The bundles tell you which renovation packages support which rent bumps.
4. Cross-reference with Section 9 "WTP Calculator" to break down which single features drive the bundle's value.
5. Decision rule: if a repositioning bundle preserves or grows share at the new rent, your WTP supports the upside. Move bundle into the underwriting model.

### Workflow B — Capture-Rate Underwriting
1. Set Highland Square to its underwritten lease-up specification.
2. Set the competitor to a market-weighted average of the 3-4 closest comps.
3. Adjust population weights in Section 6 to match the trade-area demographic mix (use ACS B25010, B19001 for income, B25127 for renter age distribution).
4. The population-weighted share is your conditional capture rate.
5. **Capture rate × qualified renter pool size = annual signed leases.** Compare to your underwriting velocity.

### Workflow C — Rent Sensitivity
1. Fix all Highland Square attributes except price.
2. Manually change Highland Square's Price dropdown across the 4 levels.
3. After each change, hit "Run Simulation" and note the share.
4. Plot price vs. share — the slope is your demand elasticity at the current positioning.
5. Use this for stress-testing: if rent has to drop to hit lease-up velocity, what's the rent floor?

### Caveats for the IC memo
- Shares are **conditional** on choosing to lease (no outside-option data). Combine with external market sizing for absolute counts.
- The LLM-as-respondent design has known position-bias and helpful-assistant tendencies. Reported WTP figures should be treated as **directional and ordinal**, with magnitudes triangulated against industry benchmarks (NMHC 2024 renter preferences, ALN concession data, CoStar comparable rents).
- Per-persona weights are placeholder; replace with verified demographic data for the trade area before final numbers go to committee.